## 1. Import Libraries

In [120]:
# Load libraries
import os, sys #os dung de doc duong dan file, tao xoa thu muc kiem tra file ton tai,sys lay tham so dong lenh vaf dung thoat chuong trinh
from IPython import display # dung cho viec hien hinh anh
import numpy as np
import matplotlib.pyplot as plt # ve do thi 
import pandas as pd # dung su ly du lieu dang bang doc , loc du lieu tu csv excel
import seaborn as sns # ve bieu do nang cao
import joblib # luu va load mo hinh machine learning

from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder # chuyen du lieu thanh vecto ,text thanh so, du lieu thu tu
from sklearn.preprocessing import MinMaxScaler, StandardScaler # chuan hoa du lieu tu 0 ->1,Chuẩn hóa dữ liệu theo mean = 0 và std = 1
from sklearn.model_selection import train_test_split # chia va train/ test

import warnings

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams['figure.dpi'] = 100

warnings.filterwarnings("ignore")

## 2. Load Dataset

In [121]:
data_path = "data/pima-indians-diabetes.csv"

data_names = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome"
]

df_dataset = pd.read_csv(data_path, names=data_names)

## 3. Phân tích dữ liệu (Analyze Data)

In [122]:
# shape
print(f'+ Shape : {df_dataset.shape}')
# types
print(f'+ Data Types: \n{df_dataset.dtypes}')
# head, tail
print(f'+ Contents: ')
display.display(df_dataset.head(5))
display.display(df_dataset.tail(5))
# info
df_dataset.info()

+ Shape : (768, 9)
+ Data Types: 
Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object
+ Contents: 


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1
767,1,93,70,31,0,30.4,0.315,23,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [123]:
description = df_dataset.describe().T
display.display(description)

,count,mean,std,min,25%,50%,75%,max
Pregnancies,768.0,3.845052,3.369578,0.000,1.00000,3.0000,6.00000,17.00
Glucose,768.0,120.894531,31.972618,0.000,99.00000,117.0000,140.25000,199.00
BloodPressure,768.0,69.105469,19.355807,0.000,62.00000,72.0000,80.00000,122.00
SkinThickness,768.0,20.536458,15.952218,0.000,0.00000,23.0000,32.00000,99.00
Insulin,768.0,79.799479,115.244002,0.000,0.00000,30.5000,127.25000,846.00
BMI,768.0,31.992578,7.884160,0.000,27.30000,32.0000,36.60000,67.10
DiabetesPedigreeFunction,768.0,0.471876,0.331329,0.078,0.24375,0.3725,0.62625,2.42
Age,768.0,33.240885,11.760232,21.000,24.00000,29.0000,41.00000,81.00
Outcome,768.0,0.348958,0.476951,0.000,0.00000,0.0000,1.00000,1.00


**Nhận xét**:
+ Dữ liệu có 9 tính chất để phân lớp: Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age, Outcome
+ Giá trị của từng tính chất lần lượt:số lần mang thai, mg/dL, mmHg, mm, µU/mL, kg/m², giá trị càng lờn càng bị
+ Tổng số dòng dữ liệu là 768 dòng
+ Dữ liệu để phân lớp ở cột Outcome

#### (2) **Kiểm tra tính toàn vẹn của dữ liệu**
+ Dữ liệu có bị trùng lặp không? Hiển thị dòng bị vi phạm.
+ Dữ liệu có tồn tại giá trị Null không? Hiển thị dòng bị vi phạm.
+ Dữ liệu có tồn tại giá trị NaN không? Hiển thị dòng bị vi phạm.

In [124]:
has_null = df_dataset.isnull().sum().any() # tim null, roi xem co bao nhieu gia tri null,check co cai nao lon hon 0 ko
has_nan  = df_dataset.isna().sum().any()
n_duplicated = df_dataset.duplicated().sum()
print(f'Tính toàn vẹn dữ liệu:')
print(f'+ Có giá trị Null: {has_null}')
if has_null:
    display.display(df_dataset[df_dataset.isnull().any(axis=1)])
print(f'+ Có giá trị Nan: {has_nan}')
if has_nan:
    display.display(df_dataset[df_dataset.isna().any(axis=1)])
print(f'+ Số dòng trùng: {n_duplicated}')


Tính toàn vẹn dữ liệu:
+ Có giá trị Null: False
+ Có giá trị Nan: False
+ Số dòng trùng: 0


#### (3) **Các tính chất thống kê trên dữ liệu số**


In [125]:
description = df_dataset.describe().T
display.display(description)

,count,mean,std,min,25%,50%,75%,max
Pregnancies,768.0,3.845052,3.369578,0.000,1.00000,3.0000,6.00000,17.00
Glucose,768.0,120.894531,31.972618,0.000,99.00000,117.0000,140.25000,199.00
BloodPressure,768.0,69.105469,19.355807,0.000,62.00000,72.0000,80.00000,122.00
SkinThickness,768.0,20.536458,15.952218,0.000,0.00000,23.0000,32.00000,99.00
Insulin,768.0,79.799479,115.244002,0.000,0.00000,30.5000,127.25000,846.00
BMI,768.0,31.992578,7.884160,0.000,27.30000,32.0000,36.60000,67.10
DiabetesPedigreeFunction,768.0,0.471876,0.331329,0.078,0.24375,0.3725,0.62625,2.42
Age,768.0,33.240885,11.760232,21.000,24.00000,29.0000,41.00000,81.00
Outcome,768.0,0.348958,0.476951,0.000,0.00000,0.0000,1.00000,1.00


#### (4) **Tần số xuất hiện (Distribution) trên dữ liệu phân lớp (Outcome) và dữ liệu danh mục (Category)**


In [126]:
df_dataset["Outcome"].value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

## Kiểm tra dữ liệu

xóa dữ liệu trùng lập

In [127]:
duplicated = df_dataset.duplicated()

print(f"Số dòng bị trùng: {duplicated.sum()}")

df_clean = df_dataset.drop_duplicates()

print(f"Kích thước sau khi xóa dòng trùng: {df_clean.shape}")

Số dòng bị trùng: 0
Kích thước sau khi xóa dòng trùng: (768, 9)


In [128]:
physiological_ranges = {
    "Pregnancies": (0, 20),
    "Glucose": (20, 200),
    "BloodPressure": (20, 200),
    "SkinThickness": (20, 200),
    "Insulin": (16, 200),
    "BMI": (10, 80),
    "DiabetesPedigreeFunction": (0, 2.5),
    "Age": (21, 120)
}

In [129]:
def detect_physiological_errors(df, physiological_ranges):
    errors = {}
    cols_no_zero = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
    ]
    for col, (min_val, max_val) in physiological_ranges.items():
        if col not in df.columns:
            continue
        
        # Nếu KHÔNG cho phép = 0
        if col in cols_no_zero:
            invalid_mask = (df[col] <= 0) | (df[col] < min_val) | (df[col] > max_val)
        else:
            # Cho phép = 0
            invalid_mask = (df[col] < min_val) | (df[col] > max_val)

        invalid_values = df.loc[invalid_mask, col]

        if not invalid_values.empty:
            errors[col] = {
                "count": invalid_mask.sum(),
                "min_actual": df[col].min(),
                "max_actual": df[col].max(),
                "problem_values": invalid_values.unique().tolist()
            }

    return errors

In [130]:
errors= detect_physiological_errors(df_dataset,physiological_ranges)
print ("====Dữ liệu lỗi được phát hiện====")
for col, info in errors.items():
    print(f"{col} có: {info['count']} giá trị lỗi")
    print(f"- Range thực tế: {info['min_actual']}-{info['max_actual']}")
    print(f"- Giá trị có vấn đề  {info['problem_values']}")

====Dữ liệu lỗi được phát hiện====
Glucose có: 5 giá trị lỗi
- Range thực tế: 0-199
- Giá trị có vấn đề  [0]
BloodPressure có: 35 giá trị lỗi
- Range thực tế: 0-122
- Giá trị có vấn đề  [0]
SkinThickness có: 338 giá trị lỗi
- Range thực tế: 0-99
- Giá trị có vấn đề  [0, 19, 15, 11, 18, 10, 13, 14, 17, 12, 16, 7, 8]
Insulin có: 461 giá trị lỗi
- Range thực tế: 0-846
- Giá trị có vấn đề  [0, 543, 846, 230, 235, 245, 207, 240, 300, 342, 304, 270, 228, 220, 495, 225, 325, 284, 204, 485, 285, 210, 318, 280, 271, 478, 744, 370, 680, 402, 258, 375, 278, 545, 360, 215, 205, 231, 255, 249, 293, 465, 415, 275, 579, 310, 474, 277, 14, 237, 328, 250, 480, 265, 326, 274, 330, 600, 272, 321, 15, 440, 540, 335, 387, 291, 392, 510]
BMI có: 11 giá trị lỗi
- Range thực tế: 0.0-67.1
- Giá trị có vấn đề  [0.0]


In [131]:
def handle_zero_values(df):
    """
    Xử lý các giá trị 0 bất thường trong các chỉ số sinh học
    bằng cách thay thế bằng median (không tính giá trị 0)
    """
    df=df.copy()
    zero_sensitive_columns = [
        'Glucose',
        'BloodPressure',
        'SkinThickness',
        'Insulin',
        'BMI'
    ]

    for col in zero_sensitive_columns:
        # Bỏ qua nếu cột không tồn tại
        if col not in df.columns:
            continue

        # Tạo mask cho giá trị = 0
        zero_mask = df[col] == 0

        if zero_mask.any():
            count_zero = zero_mask.sum()
            print(f"Phát hiện {count_zero} giá trị 0 trong cột '{col}'")

            # Tính median (chỉ lấy giá trị > 0)
            median_val = df.loc[df[col] > 0, col].median()

            # Tránh trường hợp toàn bộ cột = 0
            if pd.isna(median_val):
                print(f"→ Bỏ qua '{col}' vì không có giá trị hợp lệ để tính median")
                continue

            # Thay thế
            df.loc[zero_mask, col] = median_val

            print(f"→ Đã thay bằng median: {median_val:.2f}")

    return df


# Áp dụng
df_final = handle_zero_values(df_clean)

Phát hiện 5 giá trị 0 trong cột 'Glucose'
→ Đã thay bằng median: 117.00
Phát hiện 35 giá trị 0 trong cột 'BloodPressure'
→ Đã thay bằng median: 72.00
Phát hiện 227 giá trị 0 trong cột 'SkinThickness'
→ Đã thay bằng median: 29.00
Phát hiện 374 giá trị 0 trong cột 'Insulin'
→ Đã thay bằng median: 125.00
Phát hiện 11 giá trị 0 trong cột 'BMI'
→ Đã thay bằng median: 32.30


dữ liêu lỗi sau khi xử lý

In [132]:
errors= detect_physiological_errors(df_final,physiological_ranges)
print ("====Dữ liệu lỗi được phát hiện====")
for col, info in errors.items():
    print(f"{col}:{info['count']} giá trị lỗi")
    print(f"- Range thực tế: {info['min_actual']}-{info['max_actual']}")
    print(f"- Giá trị có vấn đề  {info['problem_values']}")

====Dữ liệu lỗi được phát hiện====
SkinThickness:111 giá trị lỗi
- Range thực tế: 7-99
- Giá trị có vấn đề  [19, 15, 11, 18, 10, 13, 14, 17, 12, 16, 7, 8]
Insulin:87 giá trị lỗi
- Range thực tế: 14-846
- Giá trị có vấn đề  [543, 846, 230, 235, 245, 207, 240, 300, 342, 304, 270, 228, 220, 495, 225, 325, 284, 204, 485, 285, 210, 318, 280, 271, 478, 744, 370, 680, 402, 258, 375, 278, 545, 360, 215, 205, 231, 255, 249, 293, 465, 415, 275, 579, 310, 474, 277, 14, 237, 328, 250, 480, 265, 326, 274, 330, 600, 272, 321, 15, 440, 540, 335, 387, 291, 392, 510]


In [133]:
print("=== KẾT QUẢ SAU XỬ LÝ ===")
print(f"Dữ liệu gốc: {len(df_dataset)} dòng")
print(f"Sau xử lý: {len(df_final)} dòng")
print(f"Tỷ lệ giữ lại: {len(df_final)/len(df_dataset)*100:.1f}%")

=== KẾT QUẢ SAU XỬ LÝ ===
Dữ liệu gốc: 768 dòng
Sau xử lý: 768 dòng
Tỷ lệ giữ lại: 100.0%


In [134]:
!jupyter nbconvert --to html Xu_ly_du_lieu.ipynb

[NbConvertApp] Converting notebook Xu_ly_du_lieu.ipynb to html
[NbConvertApp] Writing 323301 bytes to Xu_ly_du_lieu.html
